# Attention mechanics: masks, heads and rotary positions

This notebook builds attention from NumPy operations, checks invariants and then compares the reference calculation with PyTorch. Keeping the example small makes tensor shapes and masking behavior inspectable.

In [ ]:
import numpy as np
import pandas as pd
from nlp_portfolio.attention import (scaled_dot_product_attention, causal_mask,
                                     multi_head_attention, rotary_embedding)

x = np.array([[1., 0.], [0., 1.], [1., 1.]])
unmasked_output, unmasked_weights = scaled_dot_product_attention(x, x, x)
causal_output, causal_weights = scaled_dot_product_attention(x, x, x, causal_mask(len(x)))
print("unmasked weights:\n", np.round(unmasked_weights, 3))
print("causal weights:\n", np.round(causal_weights, 3))
assert np.allclose(causal_weights.sum(axis=-1), 1)
assert np.allclose(np.triu(causal_weights, 1), 0)

The (1/\sqrt{d_k}) factor prevents dot products from growing with head dimension. The causal mask is applied before softmax, forcing all future-token probabilities to zero.

In [ ]:
rng = np.random.default_rng(42)
tokens, model_dim, heads = 5, 8, 2
hidden = rng.normal(size=(tokens, model_dim))
projections = [rng.normal(scale=0.2, size=(model_dim, model_dim)) for _ in range(4)]
multi_output, head_weights = multi_head_attention(
    hidden, *projections, num_heads=heads, mask=causal_mask(tokens)
)
print("output shape:", multi_output.shape)
print("attention shape [heads, query, key]:", head_weights.shape)

## Rotary positional embeddings

RoPE rotates feature pairs by a position-dependent angle. Relative offsets influence query-key dot products without adding a separate positional vector to the residual stream.

In [ ]:
rotated = rotary_embedding(np.ones((6, 8)))
pd.DataFrame(rotated[:3]).round(3)

## PyTorch cross-check

An implementation test compares the NumPy output with PyTorch scaled dot-product attention. Agreement catches common errors in scaling, mask polarity and softmax dimension.

In [ ]:
import torch
from torch.nn.functional import scaled_dot_product_attention as torch_attention

q = rng.normal(size=(4, 6))
k = rng.normal(size=(4, 6))
v = rng.normal(size=(4, 3))
expected, _ = scaled_dot_product_attention(q, k, v, causal_mask(4))
observed = torch_attention(
    torch.tensor(q, dtype=torch.float64),
    torch.tensor(k, dtype=torch.float64),
    torch.tensor(v, dtype=torch.float64),
    is_causal=True,
).numpy()
print("maximum absolute difference:", np.max(np.abs(expected - observed)))
assert np.allclose(expected, observed)

In [ ]:
lengths = np.array([128, 512, 2048, 8192])
complexity = pd.DataFrame({
    "sequence_length": lengths,
    "attention_score_elements": lengths**2,
    "fp16_score_memory_MiB_per_head": lengths**2 * 2 / 1024**2,
})
complexity

The quadratic score matrix explains why long-context systems use FlashAttention, windowed attention or retrieval. FlashAttention reduces memory traffic and avoids materializing the complete matrix, while exact dense attention still has quadratic arithmetic cost.